In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate


llm = ChatMistralAI(
    model='mistral-small-latest',
    temperature=0
)

In [3]:
SUPPORT_MESSAGE = (
    "Hi, I was charged twice for my subscription this month and I really "
    "need this fixed before Friday because I'm switching banks. This is "
    "the second time this has happened and I'm getting frustrated."
)

In [ ]:

# Zero Shot Prompting
zero_shot_prompting_template = PromptTemplate(template="""Look at this customer message and tell me what is going on: 
"{support_message}"
""")

chain = zero_shot_prompting_template | llm 
result = chain.invoke({
    'support_message': SUPPORT_MESSAGE
})
print(result.content)

This customer is **frustrated and urgent** about being **double-charged** for their subscription this month. They mention it's the **second time** this has happened, which suggests a **recurring billing issue** with their payment processor or subscription service.

Key points:
- **Problem**: Duplicate charges for their subscription.
- **Urgency**: Needs resolution **before Friday** (likely due to switching banks).
- **Frustration**: This has happened before, indicating a **systemic issue** rather than a one-time error.
- **Risk**: If unresolved, they may lose trust in the service or face financial inconvenience.

**What’s likely happening?**
- A **billing glitch** (e.g., failed retry logic, API error, or misconfigured subscription settings).
- Possible **payment processor issue** (e.g., Stripe, PayPal, etc., failing to properly void a failed charge).
- The customer may have **cancelled their old card** without updating payment info, leading to retries.

**Next steps for the business:**

In [ ]:
zero_shot_prompting_with_structure = PromptTemplate(template=""" 
Classify the following customer support message.

Message: "{support_message}"
Respond with explicit three lines: 
Category: <Billing | Technical | Account | Other> 
Urgency: <Low | Medium | High> 
Reason: <one statment>
""")

chain = zero_shot_prompting_with_structure | llm
result = chain.invoke({
    'support_message': SUPPORT_MESSAGE
})
print(result.content)

Category: Billing
Urgency: High
Reason: Customer reports duplicate charges and time-sensitive bank switch.


In [5]:
examples = [
    {
        "message": "The app crashes every time I open the camera.",
        "reasoning": "This is a functional bug, not billing or account related. No deadline or repetition mentioned, so urgency is moderate.",
        "output": """{{"category": "Technical", "urgency": "Medium", "reason": "App crash is a functional bug with no stated deadline."}}"""
    },
    {
        "message": "I can't log in and I have a client demo in 10 minutes!",
        "reasoning": "Login issue is Account-related. There is an explicit, imminent deadline, so urgency is high.",
        "output": """{{"category": "Account", "urgency": "High", "reason": "Login blocker with an imminent deadline."}}"""
    }
]

example_formatter = PromptTemplate(
    template="Message: {message}\nReasoning: {reasoning}\nOutput: {output}",
    input_variables=["message", "reasoning", "output"]
)

few_shot_prompting = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_formatter,
    prefix="""
You are a support-ticket triage assistant. Classify messages using the
categories: Billing, Technical, Account, Other.

Think step by step about (a) what the customer is describing, (b) whether
they mention repetition or a deadline (signals of higher urgency), then
output ONLY a JSON object — no other text.
""",
    suffix="""
Now classify this message:
Message: "{support_message}"
Reasoning: <your step-by-step reasoning here>
Output: <JSON object with keys category, urgency, reason>
""",
    input_variables=["support_message"],
    example_separator="\n\n"
)

# print(
#     few_shot_prompting.invoke(
#         {"support_message": SUPPORT_MESSAGE}
#     )
# )
chain = few_shot_prompting | llm 
result = chain.invoke({"support_message": SUPPORT_MESSAGE})
print(result.content)

Reasoning:
(a) The customer mentions being charged twice for their subscription, which is a billing-related issue.
(b) They explicitly state a deadline ("before Friday") and mention this has happened twice before, indicating repetition and higher urgency.

Output: {"category": "Billing", "urgency": "High", "reason": "Duplicate charge with an explicit deadline and repeated occurrence."}
